# Memory and State Management in AI Agents

**Author:** Shuvam Banerji Seal

**Last Updated:** April 2026

## Learning Objectives

By the end of this notebook, you will understand:

1. Types of agent memory systems
2. State management strategies
3. Trade-offs between memory approaches
4. Implementation patterns
5. When to use each approach

## References

- [AI Agent Memory Comparative Guide: RAG vs Vector Stores](https://sparkco.ai/blog/ai-agent-memory-in-2026-comparing-rag-vector-stores-and-graph-based-approaches)
- [The Architecture of Remembrance: Architectures, Vector Stores, and GraphRAG](https://mem0.ai/blog/what-is-ai-agent-memory)
- [Vector Databases vs. Graph RAG for Agent Memory](https://machinelearningmastery.com/vector-databases-vs-graph-rag-for-agent-memory-when-to-use-which/)
- [Memory for AI Agents: The Full System](https://aurahq.ai/blog/memory-for-ai-agents)

## Types of Agent Memory

### 1. **Short-Term Memory (Working Memory)**

Temporary information in current conversation

- **Duration**: Single conversation session
- **Size**: Limited (context window constraints)
- **Access**: Immediate and fast
- **Example**: Current conversation messages

```
User: "What's my favorite color?"
Agent (short-term): "You just told me it's blue"
```

### 2. **Long-Term Memory**

Information persisted across sessions

- **Duration**: Persistent across conversations
- **Size**: Very large
- **Access**: Requires retrieval
- **Example**: User preferences, historical data

```
Session 1: "My favorite color is blue"
Session 2: Agent recalls: "Your favorite color is blue"
```

### 3. **Episodic Memory**

Memories of past events and experiences

- **Content**: "What happened" information
- **Structure**: Events with timestamps
- **Example**: "You asked about X last Tuesday"

### 4. **Semantic Memory**

Knowledge about facts and concepts

- **Content**: Facts and relationships
- **Structure**: Knowledge graphs
- **Example**: "Paris is the capital of France"

## Memory Architecture Patterns

### 1. **Simple Conversation History**

Basic approach: Keep all messages

```
┌─────────────────────────────┐
│  Memory Store               │
├─────────────────────────────┤
│ [1] User: "Hello"          │
│ [2] Agent: "Hi there"      │
│ [3] User: "How are you?"   │
│ [4] Agent: "Doing great!"  │
└─────────────────────────────┘
```

**Pros**: Simple, complete context
**Cons**: Context window limits, growing costs

### 2. **Summarization**

Compress old conversations into summaries

```
Old: [10 messages about topic X]
  ↓
Summary: "User is interested in topic X with Y constraints"
```

**Pros**: Saves tokens, maintains key info
**Cons**: Info loss, requires summarization overhead

### 3. **Vector Store / RAG**

Embed information and retrieve relevant chunks

```
Document: "User's favorite color is blue, favorite food is pizza"
  ↓
Embeddings: [0.1, 0.2, 0.3, ...]
  ↓
Query: "What's my favorite color?"
  ↓
Retrieved: "User's favorite color is blue"
```

**Pros**: Scalable, semantic search, cost-effective
**Cons**: Retrieval quality, requires embeddings

### 4. **Knowledge Graphs**

Structured relationships between entities

```
User (Alice)
  ├─ likes → Color (Blue)
  ├─ likes → Food (Pizza)
  └─ worked_at → Company (TechCorp)
```

**Pros**: Structured, relationship-aware
**Cons**: Maintenance overhead, requires structure

### 5. **Hybrid Approach**

Combine multiple memory systems

```
┌──────────────────────────────────────┐
│  Agent Memory System                 │
├──────────────────────────────────────┤
│                                      │
│  Short-term:  Message History       │
│  Medium-term: Summarized Context    │
│  Long-term:   Vector Store (RAG)    │
│  Semantic:    Knowledge Graph       │
│                                      │
└──────────────────────────────────────┘
```

## Comparison: Memory Approaches

| Aspect | History | Summary | Vector Store | Knowledge Graph | Hybrid |
|--------|---------|---------|--------------|-----------------|--------|
| **Complexity** | Low | Medium | Medium | High | Very High |
| **Storage** | O(n messages) | O(summaries) | O(embeddings) | O(nodes+edges) | High |
| **Retrieval Speed** | O(1) | O(1) | O(log n) | O(hops) | Varies |
| **Semantic Quality** | Poor | Medium | High | Very High | Very High |
| **Scalability** | Limited | Medium | High | Medium | High |
| **Token Cost** | High | Low | Medium | Medium | Low |
| **Setup Effort** | Minimal | Low | Medium | High | Very High |
| **Best For** | Short sessions | Long chats | RAG systems | Complex domains | Production |
| **Update Cost** | O(1) | O(summaries) | O(new embeddings) | O(graph updates) | Mixed |

## Vector Store / RAG Implementation

Most popular modern approach:

```python
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter

# 1. Split documents into chunks
splitter = CharacterTextSplitter(chunk_size=1000)
chunks = splitter.split_documents(documents)

# 2. Create embeddings
embeddings = OpenAIEmbeddings()

# 3. Build vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# 4. Retrieve relevant chunks
results = vector_store.similarity_search(query, k=3)

# 5. Use in agent
context = "\n".join([r.page_content for r in results])
response = agent.run(f"Context: {context}\nQuery: {query}")
```

## Knowledge Graph for Semantic Memory

Store structured relationships:

```python
class KnowledgeGraph:
    def __init__(self):
        self.graph = {}  # entity -> [(relation, entity), ...]
    
    def add_fact(self, subject, relation, obj):
        if subject not in self.graph:
            self.graph[subject] = []
        self.graph[subject].append((relation, obj))
    
    def query(self, entity):
        return self.graph.get(entity, [])
    
    def get_context(self, entity, depth=1):
        # Get entity and related entities
        context = {entity: self.query(entity)}
        if depth > 1:
            for _, related in self.query(entity):
                context.update(self.get_context(related, depth-1))
        return context

# Usage
kg = KnowledgeGraph()
kg.add_fact("Alice", "likes", "blue")
kg.add_fact("Alice", "works_at", "TechCorp")
kg.add_fact("TechCorp", "located_in", "San Francisco")

context = kg.get_context("Alice", depth=2)
```

## State Management in Agents

### Simple State Dictionary

```python
class Agent:
    def __init__(self):
        self.state = {
            "conversation_id": None,
            "messages": [],
            "context": {},
            "user_id": None
        }
    
    def update_state(self, key, value):
        self.state[key] = value
    
    def get_context(self):
        return self.state["context"]
```

### Persistent State Store

```python
# Use database for persistence
class AgentStateStore:
    def __init__(self, db):
        self.db = db
    
    def save_state(self, agent_id, state):
        self.db.set(f"agent:{agent_id}", json.dumps(state))
    
    def load_state(self, agent_id):
        data = self.db.get(f"agent:{agent_id}")
        return json.loads(data) if data else {}
```

## Memory Performance Considerations

### Token Usage

```
Approach          | Per Request Tokens
─────────────────────────────────────
Full History      | ~5000 tokens
Summarized        | ~1000 tokens
Vector Store RAG  | ~2000 tokens
Knowledge Graph   | ~1500 tokens
```

### Retrieval Latency

```
Approach          | Retrieval Time
─────────────────────────────────────
Full History      | < 1ms
Summarized        | < 1ms
Vector Store      | 10-100ms
Knowledge Graph   | 50-500ms
```

### Cost per Interaction

```
Approach          | Cost / 1000 requests
─────────────────────────────────────
Full History      | ~$50
Summarized        | ~$10
Vector Store RAG  | ~$20
Knowledge Graph   | ~$15
```

## Best Practices

### 1. **Memory Hygiene**
- Regularly prune old memories
- Archive less-used information
- Clean up duplicates

### 2. **Retrieval Strategy**
- Use semantic search for relevance
- Re-rank results by importance
- Limit retrieved context

### 3. **Update Strategy**
- Consolidate redundant information
- Update facts when new info arrives
- Maintain consistency

### 4. **Privacy and Security**
- Encrypt sensitive data
- Implement access controls
- GDPR compliance (right to be forgotten)

### 5. **Monitoring**
- Track memory hit rates
- Monitor storage costs
- Measure retrieval quality

## Decision Matrix: Which Memory System?

Choose based on your needs:

| Use Case | Recommended |
|----------|-------------|
| Chatbot (short sessions) | Conversation History |
| Chatbot (long sessions) | Summarization + History |
| Customer Service | Vector Store (FAQ/docs) |
| Knowledge Management | Knowledge Graph |
| RAG System | Vector Store |
| Complex Domain | Hybrid (Graph + Vector) |
| Personal Assistant | All types (heavy hybrid) |
| Real-time Analytics | Streaming + Recent history |
| Legal/Compliance | Knowledge Graph (structured) |
| Research Assistant | Vector Store (papers/docs) |

## Exercise: Implement Hybrid Memory System

### Task
Build an agent with:
1. Short-term: Message history (last 5 messages)
2. Medium-term: Summaries of older conversations
3. Long-term: Vector store for facts
4. Semantic: Simple knowledge graph for relationships

### Requirements
- Efficient retrieval
- Proper state management
- Clear context injection
- Cost tracking

## Key Takeaways

- **Different memory types** serve different purposes
- **Vector stores/RAG** are most popular for production
- **Knowledge graphs** excel at structured relationships
- **Hybrid approaches** combine strengths of multiple systems
- **Trade-offs exist**: complexity vs. performance vs. cost
- **Choose based on use case**, not hype
- **Monitor and optimize** memory systems for your domain